In [1]:
import sys
sys.path.append("..")

import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from training import (
    prepare_dataset,
    evaluate_tool_calling_accuracy,
    extract_tool_calls_from_text,
)

/home/oscar/llm_ws/llm-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

model_id = "../models/r32_a32_lr2e-04_bs8_best"
# Define tool schema to be parsed in the chat template
bt_tool = [{
            "name": "execute_behavior_tree",
            "arguments": {
                "bt_xml_filename": "<selected_bt>.xml",
                "execution_id": "<your numeric agent id>",
                "input_parameters": { "arg1": "value1" }
            }
        }]
model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype="auto",
        device_map="auto",
    )
processor = AutoProcessor.from_pretrained(model_id)


Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1305.20it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/home/oscar/llm_ws/llm-venv/lib/python3.12/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free t

In [3]:
# Load custom chat template from .jinja file
with open("../templates/qwen3.jinja", "r", encoding="utf-8") as f:
    custom_template = f.read()
processor.chat_template = custom_template

# Prepare dataset
train_dataset, eval_dataset, raw_eval_data = prepare_dataset(
    json_path="../data/train_dataset.json",  
    system_prompt_path="../data/system_prompt.txt",  
    processor=processor,
    train_split=0.8,
    tools=bt_tool
)

Processed 1/233 examples
Processed 100/233 examples
Processed 200/233 examples

Tokenization complete!
Split: 186 train, 47 eval


In [4]:

# Custom evaluation for tool calling accuracy
print("\n Evaluating tool calling accuracy...")
eval_results = evaluate_tool_calling_accuracy(
    model, 
    raw_eval_data,
    processor,
    tools=bt_tool
)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



 Evaluating tool calling accuracy...

EVALUATING TOOL CALLING ACCURACY
Evaluated: 10/47
Evaluated: 20/47
Evaluated: 30/47
Evaluated: 40/47

EVALUATION RESULTS:
Tool Name Accuracy: 0.00%
Argument Exact Match: 0.00%
Valid JSON Rate: 0.00%
Total tool calls evaluated: 47



In [5]:
# Find where the assistant should respond
generated_chat = []
messages = raw_eval_data[0]  # Take the first example for demonstration
print(messages)
for i, msg in enumerate(messages):
    if msg["role"] == "assistant" and "tool_calls" in msg:
        # This is the point where we want the model to generate
        context_text = processor.apply_chat_template(
            generated_chat,
            tools = bt_tool,
            tokenize=False,
            add_generation_prompt=False,
            continue_final_message=False,
            enable_thinking=False
        )
        inputs = processor(context_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1024,
                temperature=0.1,  # low for deterministic evaluation
                do_sample=False,
            )

        generated_text = processor.decode(outputs[0], skip_special_tokens=False)
        # Extract the model response part from the generated text (after the context)
        model_response = generated_text[len(context_text):].strip()
        # extract tool call
        tool_calls = extract_tool_calls_from_text(model_response)
        if tool_calls:
            generated_chat.append({
                "role": "assistant",
                "tool_calls": tool_calls
            })
        else:
            generated_chat.append({
                "role": "assistant",
                "content": model_response
            })
    else:
        generated_chat.append(msg)

final_template_text = processor.apply_chat_template(
            generated_chat,
            tools = bt_tool,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False
        )

print("\n" + "="*60)
print("MODEL GENERATED TEXT:")
print("="*60)
print(final_template_text)
print("="*60)
print("ORIGINAL DATA:")
print("="*60)
print(messages)

[{'role': 'system', 'content': 'You are a Behavior Tree execution agent. Execute the appropriate BT based on the assigned task. Always include the agent ID as execution_id in tool arguments.\n\n## Available Behavior Trees\n\n[\n  {\n    "name": "following",\n    "description": "Follows a detected person using face recognition and pose tracking. Executes: Activates face recognition, announces via TTS in Spanish \'Por favor, ponte delante para que te identifique\' asking user to stand in front, waits 5 seconds, gets person pose from perception topic /people, republishes to /detected_dynamic_pose, and follows the person continuously using FollowObject action. Can loop through multiple people in queue. When to use: When user requests to follow them or another specific person.",\n    "input_ports": [\n      {\n        "name": "person_name",\n        "type": "string",\n        "description": "The name of the person to follow"\n      }\n    ]\n  },\n  {\n    "name": "get_object",\n    "descri